<a href="https://colab.research.google.com/github/AlbertPuentes/Maching_JOBs_y_HVs_1/blob/main/Jobs_Maching_HVs_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
try:
    df_ofertas = pd.read_csv('Ofertas_laborales.csv')
    df_hvs = pd.read_csv('Test_Hvs.csv')
    print("Ofertas columns:", df_ofertas.columns.tolist())
    print("Ofertas shape:", df_ofertas.shape)
    print("Ofertas head:", df_ofertas.head(2).to_dict())
    print("\nTest_Hvs columns:", df_hvs.columns.tolist())
    print("Test_Hvs shape:", df_hvs.shape)
    print("Test_Hvs head:", df_hvs.head(2).to_dict())
except Exception as e:
    print("Error:", e)

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

# Load datasets
df_ofertas = pd.read_csv('Ofertas_laborales.csv').fillna('')
df_hvs = pd.read_csv('Test_Hvs.csv').fillna('')

# Combine text for Job Offers
df_ofertas['job_text'] = (df_ofertas['job_title'] + ' ' +
                          df_ofertas['job_skill_set'] + ' ' +
                          df_ofertas['job_description']).str.lower()

# Combine text for CVs
df_hvs['cv_text'] = (df_hvs['Título_Profesional'] + ' ' +
                     df_hvs['Perfil_Profesional'] + ' ' +
                     df_hvs['Habilidades_Técnicas'] + ' ' +
                     df_hvs['Habilidades_Blandas']).str.lower()

# Simple cleaning function
def clean_text(text):
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df_ofertas['job_text'] = df_ofertas['job_text'].apply(clean_text)
df_hvs['cv_text'] = df_hvs['cv_text'].apply(clean_text)

# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer(stop_words='english')
# Fit on all text to ensure the vocabulary covers both
all_texts = df_ofertas['job_text'].tolist() + df_hvs['cv_text'].tolist()
vectorizer.fit(all_texts)

# Transform
jobs_tfidf = vectorizer.transform(df_ofertas['job_text'])
cvs_tfidf = vectorizer.transform(df_hvs['cv_text'])

# Compute Cosine Similarity
similarity_matrix = cosine_similarity(cvs_tfidf, jobs_tfidf)

# Find Top 5 for each CV
results = []
for i in range(len(df_hvs)):
    cv_name = df_hvs.iloc[i]['Nombre']
    cv_title = df_hvs.iloc[i]['Título_Profesional']

    # Get similarities for this CV
    sim_scores = similarity_matrix[i]

    # Get indices of top 5
    top_5_idx = sim_scores.argsort()[-5:][::-1]

    row_data = {
        'Candidato': cv_name,
        'Profesión del Candidato': cv_title
    }

    for rank, idx in enumerate(top_5_idx):
        job = df_ofertas.iloc[idx]
        score = sim_scores[idx] * 100

        row_data[f'Top {rank+1} - Cargo'] = job['job_title']
        row_data[f'Top {rank+1} - Empresa'] = job['company_name']
        row_data[f'Top {rank+1} - Compatibilidad (%)'] = round(score, 2)

    results.append(row_data)

# Create DataFrame and save to Excel
df_results = pd.DataFrame(results)

# Create a nice Excel file
output_file = 'Emparejamiento_HVs_Ofertas.xlsx'
writer = pd.ExcelWriter(output_file, engine='openpyxl')
df_results.to_excel(writer, index=False, sheet_name='Resultados')

# Apply some formatting
workbook = writer.book
worksheet = writer.sheets['Resultados']

# Formatting headers
from openpyxl.styles import Font, PatternFill, Alignment
header_font = Font(bold=True, color="FFFFFF")
header_fill = PatternFill(start_color="4F81BD", end_color="4F81BD", fill_type="solid")
align_center = Alignment(horizontal="center", vertical="center")

for cell in worksheet[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = align_center

# Auto-adjust column widths
for column_cells in worksheet.columns:
    length = max(len(str(cell.value)) for cell in column_cells)
    worksheet.column_dimensions[column_cells[0].column_letter].width = length + 2

writer.close()
print(f"File generated: {output_file}")

Ofertas columns: ['job_title', 'company_name', 'location', 'start_date', 'ctc', 'experience', 'posted', 'job_skill_set', 'job_description']
Ofertas shape: (4596, 9)
Ofertas head: {'job_title': {0: 'Business Development Associate', 1: 'IELTS/PTE/Spoken English Trainer'}, 'company_name': {0: 'Break The Code', 1: 'DePioneer Education Overseas'}, 'location': {0: 'Mumbai', 1: 'Delhi'}, 'start_date': {0: 'Starts\xa0Immediately', 1: 'Starts\xa0Immediately'}, 'ctc': {0: '₹ 2,00,000 - 3,00,000', 1: '₹ 3,00,000 - 3,50,000'}, 'experience': {0: '0-1 years', 1: '0-2 years'}, 'posted': {0: '3 days ago', 1: '1 week ago'}, 'job_skill_set': {0: 'Comunicación asertiva, Negociación, Ventas, CRM, Estrategia de mercado, Orientación a resultados, Trabajo en equipo, Empatía', 1: 'Enseñanza, Comunicación oral, Planificación de clases, Paciencia, Liderazgo, Evaluación, Pedagogía, Escucha activa'}, 'job_description': {0: 'Buscamos un Business Development Associate para impulsar el crecimiento comercial, gestion